In [ ]:
import os
import re
from functools import reduce
from pathlib import Path

import pandas as pd
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

import holidays
import pandas as pd
from calendar_features import add_date_features, add_holiday_features

# use spark session
spark = (
    SparkSession.builder.master("local[*]")
    .appName("GenPM-fe")

    .config("spark.executor.memory", "24g")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "8")
    .config("spark.log.level", "ERROR")
    .getOrCreate()
)

SHARED_DIR_PATH = Path(f"/home/{USER}/app/apps/apps/generator/data/shared_dir")
EDA_DATA_PATH = SHARED_DIR_PATH / "eda_data"

raw_pm_path = EDA_DATA_PATH / "raw_pm_data"
pm_kpi_pivot = EDA_DATA_PATH / "pm_data_pivot"
sample_path = EDA_DATA_PATH / "sample"
pm_stats_path = EDA_DATA_PATH / "stats"
pm_agg_path = EDA_DATA_PATH / "agg"
pm_metadata = EDA_DATA_PATH / "pm_metadata"
PREPROCESSED_DATASET_PATH = SHARED_DIR_PATH / "preprocessed_dataset"

# Consts
MIN_KPI_COVERAGE = 0.9

PRESENTATION = False

In [ ]:
long_path = PREPROCESSED_DATASET_PATH / "pm_data_long"
wide_path = PREPROCESSED_DATASET_PATH / "pm_data_wide"
pm_df_long = spark.read.parquet(str(long_path))
pm_df_wide = spark.read.parquet(str(wide_path))

simple_reports_df = spark.read.parquet(
    str(pm_metadata / "simple_reports")
)
kpi_defs = spark.read.parquet(
    str(pm_metadata / "kpis_definitions")
)

In [ ]:
# import re

# BASE_KPI_REGEX = r"^([A-Z0-9_]+)[a-z]$"

# kpis = [r.kpi_id for r in pm_df_long.select("kpi_id").distinct().collect()]

# matched = [kpi for kpi in kpis if kpi is not None and re.match(BASE_KPI_REGEX, kpi)]
# print(matched)

In [ ]:
# kpi_bts_combos = kpis = [r.kpi_id for r in pm_df_long.select("kpi_id", "bts_id").distinct().collect()]
# len(kpi_bts_combos)

In [ ]:
# kpi_bts_cell_combos = kpis = [r.kpi_id for r in pm_df_long.select("kpi_id", "bts_id", "distname").distinct().collect()]
# len(kpi_bts_cell_combos)

In [ ]:
# len(kpis)

In [ ]:
df_features = add_date_features(df= pm_df_long, time_col= "start_time", features=["year",
                                                                          "month",
                                                                          "day_of_week",
                                                                          "day_of_year",
                                                                          "week_of_year",
                                                                          "quarter",
                                                                          "hour",
                                                                          "is_weekend"]
                                                                          )
# df_features.show()

In [ ]:
df_features_hol = add_holiday_features(df= df_features, time_col = "start_time", spark= spark, features= ["is_holiday",
                                                                                                          "is_holiday_eve",
                                                                                                          "is_long_weekend"]
                                                                                                          )
# df_features_hol.show()

# Imputing

In [ ]:
raw_pm = spark.read.parquet(str(raw_pm_path))

In [ ]:
# pm_df_long.where((f.col("kpi_id").isNull()) & (f.col("distname").isNull())).count()

In [ ]:
# pm_df_long_clean = pm_df_long.where(f.col("kpi_id").isNotNull())

In [ ]:
# pm_df_long_clean.where(f.col("kpi_value").isNull()).count()

In [ ]:
null_pct_df = (
    pm_df_long_clean
    .groupBy("kpi_id", "bts_id", "distname")
    .agg(
        f.count("*").alias("total_rows"),
        f.sum(f.col("kpi_value").isNull().cast("int")).alias("null_count")
    )
    .withColumn("null_pct", f.round((f.col("null_count") / f.col("total_rows")) * 100, 2))
    .orderBy("null_pct", ascending=False)
)

# null_pct_df.show()